<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# A minimum wage without a test

**[A minimum wage without a test](https://learning.nextia-ai.com/courses/experiments/m09/a-minimum-wage-without-a-test/)** · Experiments and Causal Impact: Did Our Change Work? · Case study 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** estimate the effect of a real policy that nobody randomised: New Jersey's minimum-wage rise of April 1992, on jobs in fast-food-style restaurants, with Pennsylvania as the comparison. You will compare seven methods under one measure (their error on fake dates, where the true effect is zero) and say what the data can and cannot tell.

| | |
|---|---|
| **Time** | About 2.5 hours with the lesson page; the notebook runs in under a minute after the install |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | Difference-in-differences and parallel trends, from [Difference-in-differences](https://learning.nextia-ai.com/courses/experiments/m05/difference-in-differences/). Synthetic control and its placebo test, from [Synthetic control](https://learning.nextia-ai.com/courses/experiments/m05/synthetic-control/). Placebo tests of your own answer, from [Test your own answer](https://learning.nextia-ai.com/courses/experiments/m06/test-your-own-answer/). |
| **Data** | US Bureau of Labor Statistics, Quarterly Census of Employment and Wages, 1990–1993 (public domain): our 0.5 MB extract for New Jersey and Pennsylvania |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m09/a-minimum-wage-without-a-test/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M09-L03-a-minimum-wage-without-a-test/a-minimum-wage-without-a-test-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scipy==1.15.3 statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: get the data

Run the next cell. It makes a new folder `minimum-wage`, puts two files in it and checks the data's checksum:

- `qcew_nj_pa_restaurants_1990_1993.csv` (0.5 MB): jobs by county and quarter in New Jersey and Pennsylvania, 1990–1993, from the US Bureau of Labor Statistics. We cut it from about 760 MB of official files.
- `extract_qcew.py`: the exact script that made the file from the official files.

The cell uses the copies next to the notebook if they are there. Otherwise it downloads them from the course's labs repository. It needs no account and takes a few seconds. You should see `Ready: qcew_nj_pa_restaurants_1990_1993.csv (5360 rows).`

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `minimum-wage` (in Colab: the **Files** panel on the left), then run the cell again.

**Source and licence.** US Bureau of Labor Statistics, Quarterly Census of Employment and Wages, quarterly files by industry, 1990–1993. Public domain: BLS says that "everything that we publish … is in the public domain" ([copyright information](https://www.bls.gov/opub/copyright-information.htm)), and asks to be cited as the source. The extract keeps New Jersey and Pennsylvania, private employers and four industries. It changes no values.

In [ ]:
# Setup: put the data and the extraction script into minimum-wage/ and check the data.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

RAW = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/M09-L03-a-minimum-wage-without-a-test/"
FILES = {"qcew_nj_pa_restaurants_1990_1993.csv": "49712711a6659530379ece53a8dbe6f873584261becd46b5a39f777279f5d38b", "extract_qcew.py": None}
HERE = Path.cwd()
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("minimum-wage").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()

def download(url, p):
    try:
        urllib.request.urlretrieve(url, p)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(p), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    if path.exists() and (expected is None or sha256(path) == expected):
        continue
    if (HERE / name).exists():  # a copy next to the notebook
        shutil.copy(HERE / name, path)
    else:
        download(RAW + name, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder minimum-wage and run again.")
os.chdir(PROJECT)
print(f"Ready: qcew_nj_pa_restaurants_1990_1993.csv ({sum(1 for _ in open('qcew_nj_pa_restaurants_1990_1993.csv')) - 1} rows).")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

### Optional: rebuild the extract yourself

The file came from four official yearly files of about 190 MB each. The next cell can rebuild it with `extract_qcew.py`: it downloads one year at a time, keeps the New Jersey and Pennsylvania rows, and deletes the year's file. It is off by default. To run it, change `REBUILD = False` to `REBUILD = True`. You should then see `same bytes as the course file: True`. If BLS has revised an old file, you may see `False` and a note: then your numbers may differ a little.

In [ ]:
# Optional: rebuild the extract from the official BLS files. About 760 MB of downloads, one year at a time.
REBUILD = False  # change to True to run it (it takes 2-10 minutes)
if REBUILD:
    import sys
    subprocess.run([sys.executable, "extract_qcew.py", "rebuilt.csv", "--download", "1990", "1991", "1992", "1993"], check=True)
    print("same bytes as the course file:", sha256(Path("rebuilt.csv")) == FILES["qcew_nj_pa_restaurants_1990_1993.csv"])
else:
    print("Skipped. Set REBUILD = True to rebuild the extract from BLS.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data

One row is one area (a state, or one of its counties), one industry and one quarter. `month1_emplvl` to `month3_emplvl` are the jobs in each month of the quarter: the people on the payroll in the pay period that includes the 12th. We use their average, `jobs`.

| `industry_code` | Industry |
|---|---|
| `722211` | limited-service restaurants (order at a counter, pay first): the closest to fast food |
| `7221` | full-service restaurants |
| `722` | all food services and drinking places |
| `10` | all private jobs (the whole economy, without government) |

`t` counts the quarters: 0 is 1990 Q1, 9 is 1992 Q2, the first quarter with New Jersey's new minimum wage.

In [ ]:
START = 9          # 1992 Q2: the first quarter with New Jersey's new minimum wage (t = 0 is 1990 Q1)
FAKE = [4, 5]      # 1991 Q1 and Q2: fake dates for the placebo tests
EAST = ["42017", "42029", "42045", "42091", "42101", "42077", "42095"]  # eastern Pennsylvania (7 counties)

def label(t):
    return f"{1990 + t // 4} Q{t % 4 + 1}"

raw = pd.read_csv("qcew_nj_pa_restaurants_1990_1993.csv", dtype={"area_fips": str, "industry_code": str, "disclosure_code": str})
raw["jobs"] = raw[["month1_emplvl", "month2_emplvl", "month3_emplvl"]].mean(axis=1)  # the quarter's average
raw["t"] = (raw["year"] - 1990) * 4 + raw["qtr"] - 1
raw["state"] = np.where(raw["area_fips"].str.startswith("34"), "NJ", "PA")
raw["county"] = ~raw["area_fips"].str.endswith("000")
print("rows:", len(raw), "| industries:", sorted(raw["industry_code"].unique()), "|", label(0), "to", label(15))

> **Check.** You should see `rows: 5360 | industries: ['10', '722', '7221', '722211'] | 1990 Q1 to 1993 Q4`.

BLS does not show a county when its numbers could reveal one employer. Those rows are simply missing. We keep the counties with all 16 quarters, and we check how much of each state they cover.

In [ ]:
def county_panel(industry):
    """Counties with all 16 quarters shown and no zero: the units of the analysis."""
    p = raw[(raw["industry_code"] == industry) & raw["county"]].copy()
    full = p.groupby("area_fips")["jobs"].transform(lambda s: len(s) == 16 and (s > 0).all())
    p = p[full].copy()
    p["y"] = np.log(p["jobs"])          # log jobs: differences are (about) % changes
    p["nj"] = (p["state"] == "NJ").astype(int)
    p["season"] = p["qtr"]
    return p.sort_values(["area_fips", "t"]).reset_index(drop=True)

def state_sums(p):
    """Jobs per state and quarter, summed over the counties in the panel."""
    return p.groupby(["state", "t"])["jobs"].sum().unstack(0)

r = county_panel("722211")   # limited-service restaurants
listed = raw[(raw["industry_code"] == "722211") & raw["county"]].groupby("state")["area_fips"].nunique()
total = raw[(raw["industry_code"] == "722211") & ~raw["county"]].set_index(["state", "t"])["jobs"].unstack(0)
share = (state_sums(r) / total).loc[:START - 1].mean()
print("counties in the file:", listed.to_dict(), "| with all 16 quarters:", r.groupby("state")["area_fips"].nunique().to_dict())
print(f"share of the state's restaurant jobs in them before the rise: NJ {share['NJ']:.1%}, PA {share['PA']:.1%}")

> **Check.** You should see `counties in the file: {'NJ': 21, 'PA': 47} | with all 16 quarters: {'NJ': 21, 'PA': 46}` and `NJ 98.5%, PA 95.8%`.

## 2. The plan before the estimates

We write the plan **before** we estimate anything:

1. **The question:** did New Jersey's rise from 4.25 to 5.05 dollars an hour (1 April 1992) change the number of jobs in limited-service restaurants?
2. **The comparison:** Pennsylvania, next door, which stayed at 4.25. Units: counties.
3. **The methods:** New Jersey before and after (the naive answer); a 2 × 2 difference-in-differences on whole years; regression difference-in-differences with all of Pennsylvania, with eastern Pennsylvania, on the restaurant share of all private jobs, and with New Jersey's own trend; a synthetic control from Pennsylvania counties with at least 300 restaurant jobs.
4. **One measure for every method:** there is no true effect to compare with. So we run each method at two **fake dates**, 1991 Q1 and 1991 Q2, on the quarters before the real rise. There, the true effect is zero. The **placebo error** is the average size of the two fake "effects". The smaller it is, the more we trust the method.

## 3. Look at the jobs

The two states' restaurant jobs, each as 100 in the year before the rise:

In [ ]:
import matplotlib.pyplot as plt

sums = state_sums(r)
index = 100 * sums / sums.loc[5:8].mean()   # 100 = the average of the year before the rise
fig, ax = plt.subplots(figsize=(8, 3.5))
for s, colour in (("NJ", "tab:orange"), ("PA", "tab:blue")):
    ax.plot(index.index, index[s], marker="o", color=colour, label={"NJ": "New Jersey", "PA": "Pennsylvania"}[s])
ax.axvline(START - 0.5, color="black", ls="--", lw=1)
ax.text(START - 0.4, ax.get_ylim()[1] - 1, "New Jersey: 4.25 -> 5.05 dollars", va="top", fontsize=9)
ax.set_xticks(range(0, 16, 2), [label(t) for t in range(0, 16, 2)])
ax.set_ylabel("Restaurant jobs (year before = 100)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
plt.tight_layout()
plt.show()

Restaurants have seasons: more jobs in summer, above all at the Jersey Shore. To see past the seasons, compare each quarter with the **same quarter a year before**. The next cell prints New Jersey's growth minus Pennsylvania's, in points, for restaurants and for all private jobs.

> **Predict.** Before the rise, should the gap be about zero? What would a gap far from zero mean?

In [ ]:
a = county_panel("10")   # all private jobs, the same counties' whole economy
yoy = np.log(sums).diff(4) * 100
yoy_all = np.log(state_sums(a)).diff(4) * 100
gap = (yoy["NJ"] - yoy["PA"]).dropna()
gap_all = (yoy_all["NJ"] - yoy_all["PA"]).dropna()
print(pd.DataFrame({"restaurants": gap.round(1), "all private jobs": gap_all.round(1)},
                   index=gap.index).rename(index=label).T.to_string())

> **Check.** In 1991 Q1, the restaurant gap is `-17.3`: New Jersey's restaurant jobs grew 17 points less than Pennsylvania's, a year before any rise.

## 4. The baseline: before and after, and the 2 × 2

In [ ]:
def pct(x):
    """A difference of logs as a % change."""
    return float(np.expm1(x) * 100)

def naive(p, start, end=16):
    """New Jersey only: the year after the start against the year before."""
    s = np.log(state_sums(p)["NJ"])
    return {"estimate": pct(s.loc[start:start + 3].mean() - s.loc[start - 4:start - 1].mean())}

def did_2x2(p, start, end=16):
    """New Jersey's change minus Pennsylvania's: whole years, so the seasons cancel."""
    s = np.log(state_sums(p))
    ch = s.loc[start:start + 3].mean() - s.loc[start - 4:start - 1].mean()
    return {"estimate": pct(ch["NJ"] - ch["PA"]), "nj": pct(ch["NJ"]), "pa": pct(ch["PA"])}

n0, d0 = naive(r, START), did_2x2(r, START)
print(f"naive, New Jersey before and after: {n0['estimate']:+.1f}%")
print(f"2 x 2: New Jersey {d0['nj']:+.1f}%, Pennsylvania {d0['pa']:+.1f}%, difference {d0['estimate']:+.1f}%")

> **Check.** You should see `naive, New Jersey before and after: +2.5%` and `2 x 2: New Jersey +2.5%, Pennsylvania +2.7%, difference -0.2%`.

## 5. Difference-in-differences as a regression

The regression uses every county and quarter. Each county's own seasons are taken out first. The errors are clustered by county, as in [Difference-in-differences](https://learning.nextia-ai.com/courses/experiments/m05/difference-in-differences/).

In [ ]:
def did_reg(p, start, end=16, trend=False):
    """County and quarter effects, each county's own seasons removed, errors clustered by county."""
    q = p[p["t"] < end].copy()
    q["post"] = (q["t"] >= start).astype(int)
    q["y"] = q["y"] - q.groupby(["area_fips", "season"])["y"].transform("mean")
    f = "y ~ nj:post + C(t)" + (" + nj:t" if trend else "")
    m = smf.ols(f, q).fit(cov_type="cluster", cov_kwds={"groups": q["area_fips"].astype("category").cat.codes})
    b, se = m.params["nj:post"], m.bse["nj:post"]
    return {"estimate": pct(b), "low": pct(b - 1.96 * se), "high": pct(b + 1.96 * se), "p": float(m.pvalues["nj:post"]),
            "counties": int(q["area_fips"].nunique())}

d1 = did_reg(r, START)
print(f"regression DiD, all PA ({d1['counties']} counties): {d1['estimate']:+.1f}% "
      f"(95% {d1['low']:+.1f}% to {d1['high']:+.1f}%, p = {d1['p']:.3f})")

> **Check.** You should see `regression DiD, all PA (67 counties): -3.3% (95% -6.2% to -0.3%, p = 0.031)`.

The interval does not contain zero. Is that an effect? A **placebo test** asks the same method about a date when nothing happened: a fake rise one year early, using only the quarters before the real rise.

> **Predict.** Will the fake rises give about zero?

In [ ]:
before = r[r["t"] < START]          # only the quarters before the real rise
for t in FAKE:
    fake = did_reg(before, t, START)
    print(f"placebo, a fake rise in {label(t)}: {fake['estimate']:+.1f}% (95% {fake['low']:+.1f}% to {fake['high']:+.1f}%)")

> **Check.** You should see `placebo, a fake rise in 1991 Q2: -7.0% (95% -9.9% to -4.0%)` on the second line.

The method finds a clear "effect" where there was none. So its interval is too narrow for this question: the counties of one state share their state's ups and downs, and clustering by county cannot see that.

## 6. Three other comparisons

- **Eastern Pennsylvania** only: 7 counties around Philadelphia and the Lehigh Valley, next to New Jersey.
- **The share of private jobs:** restaurant jobs relative to all private jobs in the same county. It takes out each county's whole economy.
- **New Jersey's own trend:** a straight line through New Jersey's gap before the rise, extended after it.

In [ ]:
share_panel = r.merge(a[["area_fips", "t", "y"]], on=["area_fips", "t"], suffixes=("", "_all"))
share_panel["y"] = share_panel["y"] - share_panel["y_all"]   # restaurant jobs relative to all private jobs
east = r[(r["state"] == "NJ") | r["area_fips"].isin(EAST)]
for name, p, kw in (("eastern PA", east, {}), ("share of private jobs", share_panel, {}), ("NJ's own trend", r, {"trend": True})):
    e = did_reg(p, START, **kw)
    print(f"regression DiD, {name:<22} {e['estimate']:+5.1f}% (95% {e['low']:+.1f}% to {e['high']:+.1f}%)")

> **Check.** You should see three lines; the first is `regression DiD, eastern PA              -2.5% (95% -5.7% to +0.8%)`.

## 7. Synthetic control

A mix of Pennsylvania counties that follows New Jersey before the rise, as in [Synthetic control](https://learning.nextia-ai.com/courses/experiments/m05/synthetic-control/). We use only counties with at least 300 restaurant jobs: small counties jump when one restaurant opens or closes.

In [ ]:
from scipy.optimize import minimize

def synth_weights(treated_pre, donors_pre):
    """The course project's synth.py: weights >= 0 that add up to 1 and follow the treated unit before."""
    k = donors_pre.shape[1]
    res = minimize(lambda w: float(((treated_pre - donors_pre @ w) ** 2).sum()), np.full(k, 1 / k), method="SLSQP",
                   bounds=[(0, 1)] * k, constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
                   options={"maxiter": 500, "ftol": 1e-12})
    w = np.clip(res.x, 0, None)
    return w / w.sum()

def synth_one(series, treated, start, end):
    """series: unit -> log jobs by quarter. Fit on t < start; the effect is the mean gap on start <= t < end."""
    donors = [u for u in series if u != treated]
    y = series[treated][:end]
    x = np.column_stack([series[u][:end] for u in donors])
    pre = np.arange(end) < start
    x_mean, y_mean = x[pre].mean(axis=0), y[pre].mean()
    w = synth_weights(y[pre] - y_mean, x[pre] - x_mean)  # match the shape, not the level
    gap = y - ((x - x_mean) @ w + y_mean)
    rmspe_pre = float(np.sqrt((gap[pre] ** 2).mean()))
    return {"weights": dict(zip(donors, map(float, w))), "estimate": pct(gap[~pre].mean()), "gap": gap,
            "rmspe_pre": rmspe_pre, "ratio": float(np.sqrt((gap[~pre] ** 2).mean())) / rmspe_pre}

def synth(p, start, end=16, min_jobs=300):
    """New Jersey against a mix of Pennsylvania counties with at least min_jobs restaurant jobs before."""
    big = p[(p["state"] == "PA") & (p["t"] < START)].groupby("area_fips")["jobs"].mean()
    series = {"NJ": np.log(state_sums(p)["NJ"].to_numpy())}
    for f in sorted(big[big >= min_jobs].index):
        series[f] = p.loc[p["area_fips"] == f, "y"].to_numpy()
    out = synth_one(series, "NJ", start, end)
    out["series"] = series
    return out

names = dict(zip(raw["area_fips"], raw["area_title"].str.replace(", Pennsylvania", "")))
sc = synth(r, START)
print(f"synthetic New Jersey from {len(sc['weights'])} donor counties: " + ", ".join(
    f"{names[k]} {v:.2f}" for k, v in sorted(sc["weights"].items(), key=lambda kv: -kv[1]) if v >= 0.01))
print(f"fit before: RMSPE {sc['rmspe_pre']:.4f}; effect after: {sc['estimate']:+.1f}%")

> **Check.** You should see `fit before: RMSPE 0.0265; effect after: -5.1%`.

The gap between New Jersey and its synthetic twin, quarter by quarter:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(range(16), 100 * sc["gap"], marker="o", color="tab:orange", label="New Jersey minus its synthetic twin")
ax.axhline(0, color="black", lw=1)
ax.axvline(START - 0.5, color="black", ls="--", lw=1)
ax.set_xticks(range(0, 16, 2), [label(t) for t in range(0, 16, 2)])
ax.set_ylabel("Gap (%)")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

Is the gap after the rise unusual? Give every donor county the same treatment: build its own synthetic twin from the other donors, and compare the gap after with the gap before (the post/pre ratio).

In [ ]:
others = {u: s for u, s in sc["series"].items() if u != "NJ"}
placebo = {u: synth_one(others, u, START, 16) for u in others}   # each donor plays New Jersey in turn
ratios = sorted([(v["ratio"], u) for u, v in placebo.items()] + [(sc["ratio"], "NJ")], reverse=True)
rank = [u for _, u in ratios].index("NJ") + 1
effects = sorted(v["estimate"] for v in placebo.values())
print(f"New Jersey's post/pre gap ratio {sc['ratio']:.1f}: rank {rank} of {len(ratios)} (p = {rank / len(ratios):.2f})")
print(f"placebo effects: {effects[0]:+.1f}% to {effects[-1]:+.1f}%; "
      f"{sum(abs(v) >= abs(sc['estimate']) for v in effects)} of {len(effects)} at least as large as New Jersey's")

> **Check.** You should see `New Jersey's post/pre gap ratio 2.7: rank 26 of 40 (p = 0.65)`.

> **Your turn: change one thing.** Build the synthetic control with donors of at least **1,000** restaurant jobs instead of 300 (`synth(r, START, min_jobs=1000)`). Predict first: will the estimate stay near −5%? Put the estimate in `synth_1000`, rounded to 1 decimal.

In [ ]:
synth_1000 = ...  # the estimate of synth(r, START, min_jobs=1000), rounded to 1 decimal

In [ ]:
expect('synth_1000', synth_1000, -6.3, tolerance=0.001)

## 8. A fair comparison: one measure for seven methods

Every method, at the real date and at the two fake dates. This takes about 10 seconds.

In [ ]:
methods = {
    "New Jersey before and after": lambda s, e: naive(r, s, e),
    "2 x 2, whole years, all PA": lambda s, e: did_2x2(r, s, e),
    "Regression DiD, all PA": lambda s, e: did_reg(r, s, e),
    "Regression DiD, eastern PA": lambda s, e: did_reg(east, s, e),
    "Regression DiD, share of private jobs": lambda s, e: did_reg(share_panel, s, e),
    "Regression DiD, NJ's own trend": lambda s, e: did_reg(r, s, e, trend=True),
    "Synthetic control": lambda s, e: synth(r, s, e),
}
rows = []
for name, f in methods.items():
    real = f(START, 16)
    fakes = [f(t, START)["estimate"] for t in FAKE]
    rows.append({"method": name, "estimate %": round(real["estimate"], 1),
                 "fake 1991 Q1 %": round(fakes[0], 1), "fake 1991 Q2 %": round(fakes[1], 1),
                 "placebo error": round(float(np.mean(np.abs(fakes))), 1)})
table = pd.DataFrame(rows).set_index("method")
print(table.to_string())

> **Check.** On the line `Regression DiD, eastern PA`, you should see the estimate `-2.5` and the placebo error `5.0`, the smallest in the table.

> **Your turn.** Compute the 2 × 2 difference-in-differences on whole years with **eastern Pennsylvania** (`did_2x2` with the panel `east`). Put its estimate in `east_2x2`, rounded to 1 decimal.

In [ ]:
east_2x2 = ...  # the 2 x 2 estimate with eastern Pennsylvania, rounded to 1 decimal

In [ ]:
expect('east_2x2', east_2x2, -0.9, tolerance=0.001)

## 9. Compare with Card and Krueger

Card and Krueger surveyed fast-food restaurants in February–March and November–December 1992. The closest quarters here are 1992 Q1 and 1992 Q4. We cannot use their data (it has no licence), but we can use the same window, New Jersey against eastern Pennsylvania:

In [ ]:
ck = np.log(state_sums(east))
ck_nj, ck_pa = ck.loc[11, "NJ"] - ck.loc[8, "NJ"], ck.loc[11, "PA"] - ck.loc[8, "PA"]
print(f"1992 Q1 to 1992 Q4: New Jersey {pct(ck_nj):+.1f}%, eastern Pennsylvania {pct(ck_pa):+.1f}%, "
      f"difference {pct(ck_nj - ck_pa):+.1f}%")

> **Check.** You should see `1992 Q1 to 1992 Q4: New Jersey +6.7%, eastern Pennsylvania +3.2%, difference +3.4%`.

## 10. The chosen method on new input

The method with the smallest placebo error was the regression with eastern Pennsylvania. We run it, unchanged, on two other industries, with the same placebo test.

In [ ]:
for industry, name in (("7221", "full-service restaurants"), ("722", "all food services and drinking places")):
    p_new = county_panel(industry)
    e_new = p_new[(p_new["state"] == "NJ") | p_new["area_fips"].isin(EAST)]
    real = did_reg(e_new, START)
    fakes = [did_reg(e_new, t, START)["estimate"] for t in FAKE]
    print(f"{name}: {real['estimate']:+.1f}% (95% {real['low']:+.1f}% to {real['high']:+.1f}%); "
          f"placebo {fakes[0]:+.1f}%, {fakes[1]:+.1f}%")

> **Check.** You should see two lines; the first is `full-service restaurants: +1.3% (95% -1.2% to +3.7%); placebo -2.0%, -1.7%`.

> **Your turn.** The placebo error of the eastern Pennsylvania method was `5.0` points. New Jersey had about `38,300` limited-service restaurant jobs in the year before the rise. How many jobs is one placebo error? Put it in `noise_jobs`, rounded to the nearest whole job. Use `table` for the error.

In [ ]:
noise_jobs = ...  # the placebo error (in %) of "Regression DiD, eastern PA" times New Jersey's jobs in the year before

In [ ]:
expect('noise_jobs', noise_jobs, 1915, tolerance=0.001)

## Next

The [lesson page](https://learning.nextia-ai.com/courses/experiments/m09/a-minimum-wage-without-a-test/) explains each step, gives the recommendation and its limits, and has the knowledge checks that count toward your certificate. Then make it a portfolio piece, as the page shows.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [The final assignment](https://learning.nextia-ai.com/courses/experiments/final-assignment/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m09/a-minimum-wage-without-a-test/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The data is real and public (US Bureau of Labor Statistics, public domain): counts of jobs, no people.